# Smart Packing List Agent

An agentic AI system that helps you build a personalised packing list by reasoning over real weather data and your trip details.

**How it works (agent loop overview)**
1. You describe a trip goal in plain English.
2. The LLM (Groq) decides which tool to call next — `check_weather` or `add_item` — based on what it already knows.
3. Python executes the tool and feeds the result back to the LLM.
4. Steps 2–3 repeat until the LLM says it is done (max 8 steps).
5. Every step is recorded in a **trace** so you can inspect exactly what happened.

> **Assignment note:** This is a *genuine agent*, not a chatbot. The packing decisions come from the LLM reasoning over raw weather JSON — no hardcoded `if rainy: add_umbrella` logic anywhere in the Python code.

---
## 1. Setup & Imports

Load environment variables from `.env`, initialise the Groq client, and set the single global configuration flag.

In [14]:
# ── standard library ─────────────────────────────────────────────────────────
import os
import json
import datetime
from typing import Any

# ── third-party ───────────────────────────────────────────────────────────────
import requests
from openai import OpenAI
from dotenv import load_dotenv

# ── load .env ─────────────────────────────────────────────────────────────────
load_dotenv()  # reads .env from the current working directory

# ── Groq client (OpenAI-compatible) ───────────────────────────────────────────
client = OpenAI(
    base_url=os.getenv("GROQ_ENDPOINT"),
    api_key=os.getenv("GROQ_API_KEY"),
)
MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")

# ── global config flag ────────────────────────────────────────────────────────
# Set True  → use OpenAI-style native tool calling (tools=[...] parameter).
# Set False → fall back to JSON parsing from system-prompted plain text output.
USE_NATIVE_TOOL_CALLING: bool = True

print(f"   Client ready  |  model: {MODEL}")
print(f"   Native tool calling: {USE_NATIVE_TOOL_CALLING}")
print(f"   GROQ_ENDPOINT : {os.getenv('GROQ_ENDPOINT')}")
print(f"   OPENWEATHER_API_KEY set: {'yes' if os.getenv('OPENWEATHER_API_KEY') else 'NO — check .env'}")

   Client ready  |  model: openai/gpt-oss-120b
   Native tool calling: True
   GROQ_ENDPOINT : https://api.groq.com/openai/v1
   OPENWEATHER_API_KEY set: yes


---
## 2. Tool Definitions

Two real Python functions that the agent can invoke:

| Tool | What it does |
|------|--------------|
| `check_weather(city)` | Calls OpenWeatherMap's free 5-day forecast API and returns the raw JSON |
| `add_item(item, reason)` | Appends an item + reason to the active trip's packing list in `SessionMemory` |

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# TOOL 1 — check_weather
# ─────────────────────────────────────────────────────────────────────────────

def check_weather(city: str) -> dict:
    """
    Fetch a 5-day / 3-hour weather forecast for *city* from OpenWeatherMap.

    Returns:
        dict — raw API JSON on success, or {"error": "...", "city": city} on failure.
    """
    api_key = os.getenv("OPENWEATHER_API_KEY", "")
    if not api_key:
        return {"error": "OPENWEATHER_API_KEY not set in environment.", "city": city}

    url = (
        f"https://api.openweathermap.org/data/2.5/forecast"
        f"?q={city}&appid={api_key}&units=metric"
    )
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()          # raises HTTPError for 4xx / 5xx
        data = resp.json()
        return data
    except requests.exceptions.HTTPError as exc:
        return {
            "error": f"HTTP {exc.response.status_code}: {exc.response.text[:200]}",
            "city": city,
        }
    except requests.exceptions.ConnectionError:
        return {"error": "Network connection failed. Check your internet.", "city": city}
    except requests.exceptions.Timeout:
        return {"error": "OpenWeatherMap request timed out.", "city": city}
    except Exception as exc:            # catch-all so the agent loop never crashes
        return {"error": str(exc), "city": city}

In [16]:
# ── Standalone test for check_weather ─────────────────────────────────────────
print("=== Standalone test: check_weather ===")
_test_weather = check_weather("London")

if "error" in _test_weather:
    print(f"     Error returned: {_test_weather['error']}")
else:
    city_name = _test_weather.get("city", {}).get("name", "?")
    count     = _test_weather.get("cnt", 0)
    first     = _test_weather["list"][0] if _test_weather.get("list") else {}
    temp      = first.get("main", {}).get("temp", "?")
    desc      = first.get("weather", [{}])[0].get("description", "?")
    print(f"     City: {city_name}  |  Forecast entries: {count}")
    print(f"     First entry → {temp} °C, {desc}")

=== Standalone test: check_weather ===
     Error returned: Network connection failed. Check your internet.


In [17]:
# ─────────────────────────────────────────────────────────────────────────────
# TOOL 2 — add_item
#
# NOTE: This function references `_active_memory`, a module-level variable that
# is set to a SessionMemory instance in Section 3.  The tool is defined here so
# its schema can be placed alongside check_weather, but it only works once a
# trip has been started.  The agent loop always calls start_trip() first.
# ─────────────────────────────────────────────────────────────────────────────

_active_memory = None   # will be replaced by a SessionMemory instance below

def add_item(item: str, reason: str = "") -> str:
    """
    Add *item* to the current trip's packing list in SessionMemory.

    Args:
        item   (str): The item to pack, e.g. "waterproof jacket".
        reason (str): Short explanation, e.g. "rain forecast Mon–Wed".

    Returns:
        str — confirmation message, or an error string if no trip is active.
    """
    if _active_memory is None or _active_memory.current_trip is None:
        return " No active trip. Call start_trip() on the memory object first."

    _active_memory.current_trip["items"].append({"item": item, "reason": reason})
    return f" Added '{item}' to packing list" + (f" (reason: {reason})" if reason else "")

In [18]:
# ── Standalone test for add_item ──────────────────────────────────────────────
# We temporarily create a minimal memory-like dict to test without the full class.
print("=== Standalone test: add_item (no active trip) ===")
result_no_trip = add_item("sunscreen", "beach holiday")
print(" ", result_no_trip)  # should warn about no active trip

# Simulate an active trip
_active_memory = type('FakeMemory', (), {'current_trip': {'items': []}})()  # tiny mock
result_with_trip = add_item("sunscreen", "sunny forecast throughout")
print("\n=== Standalone test: add_item (with active trip) ===")
print(" ", result_with_trip)
print("  items in mock trip:", _active_memory.current_trip['items'])

# Reset — real memory will be set in Section 3
_active_memory = None

=== Standalone test: add_item (no active trip) ===
   No active trip. Call start_trip() on the memory object first.

=== Standalone test: add_item (with active trip) ===
   Added 'sunscreen' to packing list (reason: sunny forecast throughout)
  items in mock trip: [{'item': 'sunscreen', 'reason': 'sunny forecast throughout'}]


---
## 3. Memory Class (`SessionMemory`)

`SessionMemory` is a plain Python class (no external DB) that lives for the whole notebook session.  
The key design requirement is that **later cells read back from the same object** — the final review step calls `memory.summary()` rather than re-deriving anything from scratch.

In [19]:
class SessionMemory:
    """
    Persists a single active trip across the entire notebook session.

    Attributes
    ----------
    current_trip : dict | None
        {
          'destination' : str,
          'days'        : int,
          'trip_type'   : str,
          'weather'     : dict | None,   # raw API response
          'items'       : list[dict]     # [{item, reason}, ...]
        }
    trip_history : list[dict]
        Archived completed trips (for multi-turn demos).
    """

    def __init__(self):
        self.current_trip: dict | None = None
        self.trip_history: list[dict]  = []

    # ── lifecycle ─────────────────────────────────────────────────────────────

    def start_trip(self, destination: str, days: int, trip_type: str) -> None:
        """
        Start a new trip, archiving any previous one.
        If a trip with the *same* destination is already active, update it
        in-place (supports the 'add a day-trip' follow-up scenario).
        """
        if (
            self.current_trip is not None
            and self.current_trip["destination"].lower() == destination.lower()
        ):
            # Same destination → update metadata, keep accumulated items & weather
            self.current_trip["days"]      = days
            self.current_trip["trip_type"] = trip_type
            print(f" Updated existing trip to {destination} (now {days} days, {trip_type})")
        else:
            # Archive previous trip if any
            if self.current_trip is not None:
                self.trip_history.append(self.current_trip)
            self.current_trip = {
                "destination" : destination,
                "days"        : days,
                "trip_type"   : trip_type,
                "weather"     : None,
                "items"       : [],
            }
            print(f" Started trip: {destination} | {days} days | {trip_type}")

    def store_weather(self, weather_data: dict) -> None:
        """Persist raw weather API result into the current trip."""
        if self.current_trip is None:
            raise RuntimeError("No active trip. Call start_trip() first.")
        self.current_trip["weather"] = weather_data

    def end_trip(self) -> None:
        """Archive current trip and clear the active slot."""
        if self.current_trip:
            self.trip_history.append(self.current_trip)
        self.current_trip = None

    # ── read-back ─────────────────────────────────────────────────────────────

    def summary(self) -> str:
        """
        Return a human-readable summary of the current trip and its packing list.
        This is the ONLY method the agent uses during the final review step —
        it does NOT re-derive anything; it reads directly from stored state.
        """
        if self.current_trip is None:
            return "No active trip."

        t = self.current_trip
        lines = [
            f"Destination : {t['destination']}",
            f"Duration    : {t['days']} day(s)",
            f"Trip type   : {t['trip_type']}",
            f"Weather fetched: {'yes' if t['weather'] and 'error' not in t['weather'] else 'no/error'}",
            "",
            f"Packing list ({len(t['items'])} item(s)):",
        ]
        for i, entry in enumerate(t["items"], 1):
            reason_str = f"  ← {entry['reason']}" if entry.get("reason") else ""
            lines.append(f"  {i:2}. {entry['item']}{reason_str}")
        return "\n".join(lines)

    def weather_summary(self) -> str:
        """
        Compact weather snapshot from stored data — used in the system prompt
        for the review step so the LLM does not need to call check_weather again.
        """
        if self.current_trip is None or self.current_trip["weather"] is None:
            return "No weather data stored."

        w = self.current_trip["weather"]
        if "error" in w:
            return f"Weather fetch failed: {w['error']}"

        entries = w.get("list", [])[:8]   # first 8 slots (~24 h)
        lines   = []
        for e in entries:
            dt   = e.get("dt_txt", "?")
            temp = e.get("main", {}).get("temp", "?")
            desc = e.get("weather", [{}])[0].get("description", "?")
            lines.append(f"  {dt}: {temp} °C, {desc}")
        return "\n".join(lines) if lines else "No forecast entries available."

    def item_names(self) -> list[str]:
        """Return just the item names — used for the LLM review prompt."""
        if self.current_trip is None:
            return []
        return [e["item"] for e in self.current_trip["items"]]

    def __repr__(self) -> str:
        trip = self.current_trip["destination"] if self.current_trip else "none"
        return f"<SessionMemory active_trip={trip!r} history={len(self.trip_history)}>"


# ── quick smoke test ──────────────────────────────────────────────────────────
print("=== Smoke test: SessionMemory ===")
_m = SessionMemory()
_m.start_trip("Manali", 4, "hiking + one formal dinner")
_active_memory = _m  # wire up add_item so we can test it
add_item("trekking poles", "hiking trails")
add_item("formal shirt", "nice dinner")
print()
print(_m.summary())
print()
print(repr(_m))

# Reset for real demos below
_active_memory = None

=== Smoke test: SessionMemory ===
 Started trip: Manali | 4 days | hiking + one formal dinner

Destination : Manali
Duration    : 4 day(s)
Trip type   : hiking + one formal dinner
Weather fetched: no/error

Packing list (2 item(s)):
   1. trekking poles  ← hiking trails
   2. formal shirt  ← nice dinner

<SessionMemory active_trip='Manali' history=0>


---
## 4. Agent Loop (`run_agent`)

This is the core of the project.  

**Design principles:**
- The LLM owns *all* packing logic — Python only routes tool calls and handles I/O.
- Native OpenAI tool calling is used by default (`USE_NATIVE_TOOL_CALLING = True`).
- A plain JSON-parsing fallback is activated when the flag is `False`.
- Every step is appended to a `trace` list for inspection.

In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# OpenAI tool schemas  (used when USE_NATIVE_TOOL_CALLING = True)
# ─────────────────────────────────────────────────────────────────────────────

TOOL_SCHEMAS: list[dict] = [
    {
        "type": "function",
        "function": {
            "name": "check_weather",
            "description": (
                "Fetch the 5-day weather forecast for a city from OpenWeatherMap. "
                "ALWAYS call this before recommending any weather-dependent items."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name, e.g. 'Manali' or 'Goa'.",
                    }
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "add_item",
            "description": (
                "Add a single item to the packing list with a short reason. "
                "Call this once per item — do NOT batch multiple items in one call."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "item": {
                        "type": "string",
                        "description": "Item to pack, e.g. 'waterproof jacket'.",
                    },
                    "reason": {
                        "type": "string",
                        "description": "One-line reason referencing weather or trip details.",
                    },
                },
                "required": ["item"],
            },
        },
    },
]

In [21]:
# ─────────────────────────────────────────────────────────────────────────────
# System prompt builders
# ─────────────────────────────────────────────────────────────────────────────

def build_system_prompt_native(trip_info: str) -> str:
    """System prompt for native tool-calling mode."""
    return f"""You are a smart travel packing assistant with access to two tools:
  • check_weather(city)        — get real weather data before suggesting weather-dependent items
  • add_item(item, reason)     — add one item to the packing list; call once per item

STRICT RULES:
1. You MUST call check_weather BEFORE adding any weather-dependent item (rain gear, sunscreen, etc.).
2. Add items one at a time using add_item; never batch them.
3. Each add_item call must include a specific reason referencing the weather or trip type.
4. After building the initial list, do one final reasoning pass over the trip type to check for missing items.
5. When you are satisfied the list is complete, respond with a plain-text summary — do NOT call any more tools.

Current trip info:
{trip_info}
"""


def build_system_prompt_fallback(trip_info: str) -> str:
    """System prompt for JSON-fallback mode."""
    return f"""You are a smart travel packing assistant.  You must output EXACTLY one JSON object per turn.

Available actions:
  {{"action": "check_weather", "city": "<city_name>"}}
  {{"action": "add_item",      "item": "<item>", "reason": "<reason>"}}
  {{"action": "finish",        "message": "<final summary text>"}}

STRICT RULES:
1. Call check_weather BEFORE any weather-dependent item.
2. Add items one at a time via add_item with a specific reason.
3. After the initial list, do one pass to check for items missing given the trip_type.
4. Use finish when done.

Output ONLY the JSON object — no markdown fences, no extra text.

Current trip info:
{trip_info}
"""


def build_review_prompt(memory: SessionMemory) -> str:
    """Prompt for the mandatory final review step — reads from memory, not re-derived."""
    return (
        f"You have built this packing list so far (read from memory):\n\n"
        f"{memory.summary()}\n\n"
        f"Weather snapshot from stored data:\n{memory.weather_summary()}\n\n"
        f"Given the trip type '{memory.current_trip['trip_type']}', "
        f"are there any important items still missing? "
        f"If yes, call add_item for each one. If the list is already complete, call finish / say you are done."
    )

In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# Tool dispatcher — maps name → Python function
# ─────────────────────────────────────────────────────────────────────────────

def dispatch_tool(name: str, args: dict, memory: SessionMemory) -> Any:
    """
    Execute the named tool with *args* and update memory side-effects.
    Returns whatever the tool returns (dict or str).
    """
    global _active_memory
    _active_memory = memory   # keep add_item wired to the right memory object

    if name == "check_weather":
        city   = args.get("city", "")
        result = check_weather(city)
        # Store raw weather in memory so the review step can read it back
        if "error" not in result:
            memory.store_weather(result)
        return result

    elif name == "add_item":
        item   = args.get("item", "")
        reason = args.get("reason", "")
        return add_item(item, reason)

    else:
        return {"error": f"Unknown tool: {name}"}


def result_summary(tool_name: str, result: Any) -> str:
    """Return a compact string representation of a tool result for the trace."""
    if tool_name == "check_weather":
        if isinstance(result, dict) and "error" in result:
            return f"ERROR: {result['error']}"
        if isinstance(result, dict) and result.get("list"):
            first = result["list"][0]
            temp  = first.get("main", {}).get("temp", "?")
            desc  = first.get("weather", [{}])[0].get("description", "?")
            cnt   = result.get("cnt", 0)
            city  = result.get("city", {}).get("name", "?")
            return f"{city}: {cnt} forecast slots, first entry = {temp}°C, {desc}"
        return str(result)[:120]
    return str(result)[:120]

In [23]:
# ─────────────────────────────────────────────────────────────────────────────
# LLM call helpers
# ─────────────────────────────────────────────────────────────────────────────

def llm_call_native(messages: list[dict]) -> Any:
    """
    Call the Groq LLM with native OpenAI tool calling.
    Returns the raw response object.
    """
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
            tool_choice="auto",
        )
        return response
    except Exception as exc:
        raise RuntimeError(f"Groq API error (native): {exc}") from exc


def llm_call_fallback(messages: list[dict], retry: bool = False) -> str:
    """
    Call the Groq LLM in JSON-fallback mode.
    Returns the raw text content of the first choice.
    On *retry* = True, a corrective message has already been appended to *messages*.
    """
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
        )
        return response.choices[0].message.content or ""
    except Exception as exc:
        raise RuntimeError(f"Groq API error (fallback): {exc}") from exc


def parse_fallback_json(text: str) -> dict | None:
    """
    Try to parse a JSON object from *text*.
    Strips accidental markdown fences if present.
    Returns None if parsing fails.
    """
    text = text.strip()
    # Strip ```json ... ``` fences
    if text.startswith("```"):
        lines = text.splitlines()
        text  = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

In [24]:
# ─────────────────────────────────────────────────────────────────────────────
# Main agent loop
# ─────────────────────────────────────────────────────────────────────────────

def run_agent(
    goal: str,
    memory: SessionMemory,
    max_steps: int = 8,
) -> tuple[str, list[tuple]]:
    """
    Run the packing-list agent for *goal*, updating *memory* in place.

    Parameters
    ----------
    goal      : str            — natural-language trip description
    memory    : SessionMemory  — shared memory object (must have start_trip called)
    max_steps : int            — safety cap on the number of LLM + tool steps

    Returns
    -------
    (final_message, trace)
      final_message : str         — the agent's concluding text
      trace         : list[tuple] — every step logged as a named tuple
    """

    trace: list[tuple] = []
    final_message      = ""

    # Build trip_info snippet for the system prompt
    t = memory.current_trip
    trip_info = (
        f"Destination: {t['destination']}\n"
        f"Duration   : {t['days']} days\n"
        f"Trip type  : {t['trip_type']}"
    )

    # ── branch on calling mode ────────────────────────────────────────────────
    if USE_NATIVE_TOOL_CALLING:
        final_message = _run_native(goal, memory, trip_info, max_steps, trace)
    else:
        final_message = _run_fallback(goal, memory, trip_info, max_steps, trace)

    trace.append(("FINAL_ANSWER", final_message))
    return final_message, trace


# ─────────────────────────────────────────────────────────────────────────────
# Native tool-calling path
# ─────────────────────────────────────────────────────────────────────────────

def _run_native(
    goal: str,
    memory: SessionMemory,
    trip_info: str,
    max_steps: int,
    trace: list,
) -> str:
    """
    Execute the agent loop using OpenAI-style native tool calling.
    Returns the final text message.
    """

    messages: list[dict] = [
        {"role": "system",    "content": build_system_prompt_native(trip_info)},
        {"role": "user",      "content": goal},
    ]

    review_done = False   # gate: have we done the mandatory review pass yet?

    for step in range(max_steps):
        response       = llm_call_native(messages)
        choice         = response.choices[0]
        finish_reason  = choice.finish_reason
        msg            = choice.message

        # Append assistant turn to history
        messages.append(msg.to_dict() if hasattr(msg, "to_dict") else
                        {"role": "assistant", "content": msg.content,
                         "tool_calls": [tc.to_dict() if hasattr(tc, "to_dict") else tc
                                        for tc in (msg.tool_calls or [])]})

        # ── no tool call → possible finish ────────────────────────────────────
        if finish_reason == "stop" or not msg.tool_calls:
            # Enforce at least one review step before accepting "done"
            if not review_done and memory.current_trip and memory.current_trip["items"]:
                review_done = True
                review_msg  = build_review_prompt(memory)
                messages.append({"role": "user", "content": review_msg})
                # Continue the loop — don't break yet
                continue
            # Review already done (or nothing added yet) — we're finished
            return msg.content or "(no final message)"

        # ── tool calls present ────────────────────────────────────────────────
        for tc in msg.tool_calls:
            tool_name = tc.function.name
            try:
                tool_args = json.loads(tc.function.arguments)
            except json.JSONDecodeError:
                tool_args = {}

            trace.append(("TOOL_CALL", tool_name, tool_args))

            result     = dispatch_tool(tool_name, tool_args, memory)
            summary    = result_summary(tool_name, result)
            trace.append(("TOOL_RESULT", tool_name, summary))

            # Serialize result for the API — weather JSON can be large; trim it
            if tool_name == "check_weather" and isinstance(result, dict) and "list" in result:
                # Send only the first 8 forecast slots to stay within token limits
                trimmed = {k: v for k, v in result.items() if k != "list"}
                trimmed["list"] = result["list"][:8]
                result_str = json.dumps(trimmed)
            else:
                result_str = json.dumps(result) if isinstance(result, dict) else str(result)

            messages.append({
                "role"        : "tool",
                "tool_call_id": tc.id,
                "name"        : tool_name,
                "content"     : result_str,
            })

    return "(max steps reached — partial list may be incomplete)"


# ─────────────────────────────────────────────────────────────────────────────
# JSON-fallback path
# ─────────────────────────────────────────────────────────────────────────────

def _run_fallback(
    goal: str,
    memory: SessionMemory,
    trip_info: str,
    max_steps: int,
    trace: list,
) -> str:
    """
    Execute the agent loop using system-prompted JSON parsing.
    Returns the final text message.
    """

    messages: list[dict] = [
        {"role": "system", "content": build_system_prompt_fallback(trip_info)},
        {"role": "user",   "content": goal},
    ]

    review_done = False

    for step in range(max_steps):
        raw_text = llm_call_fallback(messages)
        parsed   = parse_fallback_json(raw_text)

        # ── retry once on bad JSON ─────────────────────────────────────────────
        if parsed is None:
            messages.append({"role": "assistant", "content": raw_text})
            messages.append({
                "role"   : "user",
                "content": (
                    "Your last response was not valid JSON. "
                    "Output ONLY a single JSON object — no markdown, no extra text."
                ),
            })
            raw_text = llm_call_fallback(messages, retry=True)
            parsed   = parse_fallback_json(raw_text)

        if parsed is None:
            # Give up on this step; treat as finish
            return f"(JSON parse failed twice; last output: {raw_text[:200]})"

        action = parsed.get("action", "")

        # ── finish ────────────────────────────────────────────────────────────
        if action == "finish":
            if not review_done and memory.current_trip and memory.current_trip["items"]:
                review_done = True
                messages.append({"role": "assistant", "content": raw_text})
                messages.append({"role": "user", "content": build_review_prompt(memory)})
                continue
            return parsed.get("message", "(no message)")

        # ── tool call ─────────────────────────────────────────────────────────
        if action in ("check_weather", "add_item"):
            tool_args = {k: v for k, v in parsed.items() if k != "action"}
            trace.append(("TOOL_CALL", action, tool_args))

            result  = dispatch_tool(action, tool_args, memory)
            summary = result_summary(action, result)
            trace.append(("TOOL_RESULT", action, summary))

            if action == "check_weather" and isinstance(result, dict) and "list" in result:
                trimmed        = {k: v for k, v in result.items() if k != "list"}
                trimmed["list"] = result["list"][:8]
                result_str     = json.dumps(trimmed)
            else:
                result_str = json.dumps(result) if isinstance(result, dict) else str(result)

            messages.append({"role": "assistant", "content": raw_text})
            messages.append({
                "role"   : "user",
                "content": f"Tool '{action}' returned: {result_str}\n\nWhat next?",
            })
        else:
            # Unknown action — ask the model to correct itself
            messages.append({"role": "assistant", "content": raw_text})
            messages.append({
                "role"   : "user",
                "content": f"Unknown action '{action}'. Use check_weather, add_item, or finish.",
            })

    return "(max steps reached — partial list may be incomplete)"


print(" Agent loop defined.")

 Agent loop defined.


---
## 5. Demo Section

Three progressively more interesting runs:

| Run | Goal | What it tests |
|-----|------|---------------|
| A   | 4-day Manali hiking + formal dinner | weather check → mixed list → review for formal item |
| B   | 3-day Goa beach holiday | different climate, different list |
| C   | Follow-up: add a mountain day-trip to Goa | memory carries over, new items added |

The helper `print_trace` formats the trace readably.

In [25]:
# ── Pretty-print helper ────────────────────────────────────────────────────────

def print_trace(trace: list[tuple], memory: SessionMemory) -> None:
    """Print a full agent trace with emoji labels, then the final packing list."""
    print("\n" + "─" * 60)
    print(" AGENT TRACE")
    print("─" * 60)
    for step in trace:
        kind = step[0]
        if kind == "TOOL_CALL":
            _, tool, args = step
            args_str = ", ".join(f"{k}={v!r}" for k, v in args.items())
            print(f"   TOOL CALL : {tool}({args_str})")
        elif kind == "TOOL_RESULT":
            _, tool, summary = step
            print(f"   RESULT    : [{tool}] {summary}")
        elif kind == "FINAL_ANSWER":
            _, msg = step
            print(f"  LLM SAYS  : {msg[:300]}{'...' if len(msg) > 300 else ''}")
    print("─" * 60)
    print("\n FINAL PACKING LIST (from memory):")
    print(memory.summary())

In [27]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN A — Manali: 4-day hiking + one formal dinner
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print(" RUN A — Manali (4 days, hiking + formal dinner)")
print("=" * 60)

memory_a = SessionMemory()
memory_a.start_trip(
    destination = "Manali",
    days        = 4,
    trip_type   = "mostly hiking, one nice dinner planned",
)

goal_a = (
    "Pack for a 4-day trip to Manali. "
    "Most days will be hiking in the mountains. "
    "There is one nice dinner at a restaurant on the last evening."
)

final_a, trace_a = run_agent(goal_a, memory_a)
print_trace(trace_a, memory_a)

 RUN A — Manali (4 days, hiking + formal dinner)
 Started trip: Manali | 4 days | mostly hiking, one nice dinner planned

────────────────────────────────────────────────────────────
 AGENT TRACE
────────────────────────────────────────────────────────────
   TOOL CALL : check_weather(city='Manali')
   RESULT    : [check_weather] Manali: 40 forecast slots, first entry = 32.85°C, overcast clouds
   TOOL CALL : add_item(item='hiking boots (waterproof)', reason='Essential for mountain hiking and potential rain in Manali')
   RESULT    : [add_item]  Added 'hiking boots (waterproof)' to packing list (reason: Essential for mountain hiking and potential rain in Manali)
   TOOL CALL : add_item(item='moisture-wicking t‑shirts (3)', reason='Keeps you cool and dry during warm 30‑34\u202f°C hikes')
   RESULT    : [add_item]  Added 'moisture-wicking t‑shirts (3)' to packing list (reason: Keeps you cool and dry during warm 30‑34 °C hikes)
   TOOL CALL : add_item(item='quick‑dry trekking pants (2)', 

In [28]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN B — Goa: 3-day beach holiday
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print(" RUN B — Goa (3 days, beach holiday)")
print("=" * 60)

memory_b = SessionMemory()
memory_b.start_trip(
    destination = "Goa",
    days        = 3,
    trip_type   = "beach holiday",
)

goal_b = "Pack for 3 days in Goa, beach holiday."

final_b, trace_b = run_agent(goal_b, memory_b)
print_trace(trace_b, memory_b)

 RUN B — Goa (3 days, beach holiday)
 Started trip: Goa | 3 days | beach holiday

────────────────────────────────────────────────────────────
 AGENT TRACE
────────────────────────────────────────────────────────────
   TOOL CALL : check_weather(city='Goa')
   RESULT    : [check_weather] Goa: 40 forecast slots, first entry = 28.21°C, light rain
   TOOL CALL : add_item(item='lightweight waterproof rain jacket', reason='Forecast shows continuous light rain, need protection while outdoors')
   RESULT    : [add_item]  Added 'lightweight waterproof rain jacket' to packing list (reason: Forecast shows continuous light rain, need protecti
   TOOL CALL : add_item(item='compact travel umbrella', reason='Rain expected throughout stay, umbrella provides quick cover')
   RESULT    : [add_item]  Added 'compact travel umbrella' to packing list (reason: Rain expected throughout stay, umbrella provides quick cover)
   TOOL CALL : add_item(item='broad-spectrum sunscreen SPF 30', reason='Even with rain,

In [32]:
# ─────────────────────────────────────────────────────────────────────────────
# RUN C — Follow-up turn: add a mountain day-trip to the Goa holiday
#
# This reuses memory_b (same object, same trip) to prove that memory carries
# over between agent calls.  The destination stays Goa but the trip_type and
# days are updated, and the LLM must now consider mountain day-trip gear ON TOP
# of what is already in the list (it can read the existing items via memory).
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print(" RUN C — Follow-up: add a mountain day-trip to Goa holiday")
print("=" * 60)

# Show state carried over from Run B
print("\n Memory state BEFORE follow-up (carried from Run B):")
print(memory_b.summary())
print()

# Update the existing trip in-place
memory_b.start_trip(
    destination = "Goa",     # same destination → in-place update
    days        = 4,          # +1 day for the mountain trip
    trip_type   = "beach holiday + one mountain day-trip",
)

goal_c = (
    "Actually, we are adding a full day trip to the Western Ghats mountains near Goa. "
    "We'll be hiking trails for that day. "
    "Check if any new items need to be added given this change — "
    "look at what is already in the list before adding anything."
)

final_c, trace_c = run_agent(goal_c, memory_b, max_steps=10)
print_trace(trace_c, memory_b)

print("\n Memory state AFTER follow-up:")
print(memory_b.summary())

 RUN C — Follow-up: add a mountain day-trip to Goa holiday

 Memory state BEFORE follow-up (carried from Run B):
Destination : Goa
Duration    : 4 day(s)
Trip type   : beach holiday + one mountain day-trip
Weather fetched: yes

Packing list (9 item(s)):
   1. lightweight waterproof rain jacket  ← Forecast shows continuous light rain, need protection while outdoors
   2. compact travel umbrella  ← Rain expected throughout stay, umbrella provides quick cover
   3. broad-spectrum sunscreen SPF 30  ← Even with rain, UV exposure can occur; sunscreen protects skin during beach time
   4. wide-brimmed hat  ← Sun protection for beach outings and occasional clear skies
   5. UV-protection sunglasses  ← Protect eyes from sun glare during beach activities
   6. swimwear (men's board shorts)  ← Essential for beach swimming and water activities
   7. quick-dry beach towel  ← Needed for drying off after swimming; quick-dry is practical for humid weather
   8. waterproof hiking boots  ← Essential for

---
## 6. Summary

### What was built

**Two tools (real Python functions, not stubs)**

| Tool | Implementation |
|------|----------------|
| `check_weather(city)` | Calls OpenWeatherMap's free 5-day forecast REST API, returns raw JSON; all HTTP errors are caught and returned as a clean error dict so the agent loop never crashes |
| `add_item(item, reason)` | Appends `{item, reason}` to `SessionMemory.current_trip['items']`; returns a confirmation string |

**`SessionMemory` class**

A plain Python class (no database) that holds the active trip across the entire notebook session: destination, duration, trip type, raw weather API response, and the growing packing list.  
The key design point is that the **final review step calls `memory.summary()` and `memory.weather_summary()`** — it reads back the data that was stored by earlier tool calls rather than re-deriving anything, satisfying the hard requirement that memory is genuinely read back.

**Agent loop (`run_agent`)**

- Iterates up to `max_steps` times.  
- The LLM decides every action; Python only dispatches tools and plumbs results back.  
- Native OpenAI `tools=[...]` calling is the default path; a JSON-parsing fallback (with one automatic retry on malformed output) is activated by `USE_NATIVE_TOOL_CALLING = False`.  
- After the LLM first signals it is done, the loop injects a mandatory review prompt built from `memory.summary()` so the model checks for missing items given the trip type before the final answer is accepted.

---

In [30]:
my_memory = SessionMemory()
my_memory.start_trip(
    destination="Manali",
    days=5,
    trip_type="hiking trip with one formal dinner planned"
)

my_goal = "Pack for a 5-day trip to Manali, mostly hiking, one nice dinner planned"

my_result, my_trace = run_agent(my_goal, my_memory, max_steps=10)
print_trace(my_trace, my_memory)

 Started trip: Manali | 5 days | hiking trip with one formal dinner planned

────────────────────────────────────────────────────────────
 AGENT TRACE
────────────────────────────────────────────────────────────
   TOOL CALL : check_weather(city='Manali')
   RESULT    : [check_weather] Manali: 40 forecast slots, first entry = 32.85°C, overcast clouds
   TOOL CALL : add_item(item='Waterproof hiking boots', reason='Essential for rugged mountain trails and variable terrain')
   RESULT    : [add_item]  Added 'Waterproof hiking boots' to packing list (reason: Essential for rugged mountain trails and variable terrain)
   TOOL CALL : add_item(item='Moisture-wicking hiking shirts (2)', reason='Keeps you dry during warm 30°C hikes and manages sweat')
   RESULT    : [add_item]  Added 'Moisture-wicking hiking shirts (2)' to packing list (reason: Keeps you dry during warm 30°C hikes and manages sw
   TOOL CALL : add_item(item='Quick-dry hiking pants (2)', reason='Comfortable for trekking in warm, 

In [33]:
follow_up_goal = "Actually I'm extending the trip by 2 more days, add anything needed"

my_memory.start_trip(destination="Manali", days=7, trip_type="hiking trip with one formal dinner planned")

result2, trace2 = run_agent(follow_up_goal, my_memory, max_steps=10)
print_trace(trace2, my_memory)

 Updated existing trip to Manali (now 7 days, hiking trip with one formal dinner planned)

────────────────────────────────────────────────────────────
 AGENT TRACE
────────────────────────────────────────────────────────────
   TOOL CALL : check_weather(city='Manali')
   RESULT    : [check_weather] Manali: 40 forecast slots, first entry = 32.7°C, overcast clouds
   TOOL CALL : add_item(item='waterproof hiking jacket', reason='Forecast includes light rain, so a waterproof jacket is essential for staying dry on hikes')
   RESULT    : [add_item]  Added 'waterproof hiking jacket' to packing list (reason: Forecast includes light rain, so a waterproof jacket is essen
   TOOL CALL : add_item(item='waterproof hiking pants', reason='Rain is expected at night; waterproof pants will keep legs dry during wet trail conditions')
   RESULT    : [add_item]  Added 'waterproof hiking pants' to packing list (reason: Rain is expected at night; waterproof pants will keep legs dry
   TOOL CALL : add_item(i